# Workshop 03

## Usando modelos abertos

<img height="380" src="https://i.postimg.cc/FHRZ1ZVJ/Screenshot-2026-09-13-at-12-24-08.png">


## Setup

In [ ]:
# baixar algumas imagens
!wget -q "https://raw.githubusercontent.com/meta-museu-ml-workshop/WKSHP03/refs/heads/main/mm_utils_03.py" -O utils.py
!wget -q "https://raw.githubusercontent.com/meta-museu-ml-workshop/WKSHP03/refs/heads/main/data.tgz" -O data.tgz
!tar -xzf data.tgz 2> /dev/null && mv data/dataset . && mv data/*jpg . && rm -rf data.tgz data sample_data

In [ ]:
# inicializar algumas bibliotecas e outras ferramentas de Python
import numpy as np
import requests

from os import makedirs
from time import sleep
from PIL import Image as PImage

from diffusers import ControlNetModel
from diffusers import DiffusionPipeline
from diffusers import StableDiffusionControlNetPipeline as ControlNetPipeline
from torch import cuda, float16 as tf16
from transformers import pipeline
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

DEVICE = "cuda" if cuda.is_available() else "cpu"

## Processamento de Imagem

In [ ]:
DPT_MODEL = "Intel/dpt-large"

dpt = pipeline(task="depth-estimation",
               model=DPT_MODEL,
               device=DEVICE)

In [ ]:
img = PImage.open("beatles.jpg")
img.thumbnail((500, 500))

res = dpt(img)
display(img)
display(res["depth"])

In [ ]:
mask = np.array(res["depth"])
img_np = np.array(img)

ppix = (mask > 150) & (mask < 220)
img_np[~ppix] = [0,0,0]
PImage.fromarray(img_np)

## Tradução

In [ ]:
PTEN_MODEL = "unicamp-dl/translation-pt-en-t5"

tokenizer = AutoTokenizer.from_pretrained(PTEN_MODEL)
model = AutoModelForSeq2SeqLM.from_pretrained(PTEN_MODEL)

In [ ]:
texto_pt = "Eu gosto de comer arroz."

# palavras -> números
inputs = tokenizer(texto_pt, return_tensors="pt")

# traducão
outputs = model.generate(**inputs, max_new_tokens=1024)

# números -> palavras
text_en = tokenizer.decode(outputs, skip_special_tokens=True)

display(text_en)

## Geração de Imagens

In [ ]:
pipe = DiffusionPipeline.from_pretrained("stabilityai/stable-diffusion-xl-base-1.0",
                                         use_safetensors=True,
                                         safety_checker=None,
                                         torch_dtype=tf16).to(DEVICE)

In [ ]:
prompt = "Realistic photograph of an astronaut riding a glorious space horse on the moon"
out = pipe(prompt=prompt, width=512, height=512, num_inference_steps=32, guidance_scale=8.0)

display(out.images[0])

In [ ]:
# "runwayml/stable-diffusion-v1-5",
# "CompVis/stable-diffusion-v1-4",

pipe = DiffusionPipeline.from_pretrained("CompVis/stable-diffusion-v1-4",
                                         use_safetensors=True,
                                         safety_checker=None,
                                         torch_dtype=tf16).to(DEVICE)

In [ ]:
prompt = "realistic photograph of a badger against a neutral background"
out = pipe(prompt=prompt, width=512, height=512, num_inference_steps=48)
display(out["images"][0])

## Control Net

In [ ]:
controlnet = ControlNetModel.from_pretrained("lllyasviel/sd-controlnet-depth",
                                             torch_dtype=tf16)

pipe = ControlNetPipeline.from_pretrained("runwayml/stable-diffusion-v1-5",
                                          controlnet=controlnet,
                                          use_safetensors=True,
                                          safety_checker=None,
                                          torch_dtype=tf16).to(DEVICE)

In [ ]:
img = PImage.open("pessoas.jpg")

dpimg = dpt(img)["depth"]
display(dpimg)

In [ ]:
prompt = "realistic color photography. urban setting. bronze statues with pigeons on top."
negative = "cartoon, blurry, out of focus, pixelated, bad, deformed, ugly, bad anotomy"

out = pipe(prompt=prompt, image=dpimg, negative_prompt=negative,
           width=dpimg.width, height=dpimg.height,
           num_inference_steps=32,
           # guidance_scale=9.0,
           # controlnet_conditioning_scale=0.45,
          )

display(out["images"][0])



## Treinando modelos (ajuste de modelos abertos)

<img height="380" src="https://i.postimg.cc/7P3XRbLr/Screenshot-2026-09-13-at-12-24-21.png">





In [ ]:
# Instalar biblioteca YOLO
!pip install pi-heif ultralytics

In [ ]:
from ultralytics import YOLO
from utils import image_from_url
from utils import CustomizedTrainer, CustomizedValidator

In [ ]:
API_URL = "https://serpapi.com/search.json"
API_KEY = "abcd1234wxyz"

cats = ["cebolinha", "cascão", "magali", "sansão coelho", "menino maluquinho", "gudetama"]

for c in cats:
  c_path = c.replace(" ", "_")
  makedirs(f"dataset/train/{c_path}", exist_ok=True)
  makedirs(f"dataset/test/{c_path}", exist_ok=True)

  params = {
    "engine": "google_images_light",
    "q": f"{c} personagem",
    "api_key": API_KEY,
    "count": 100,
  }

  res = requests.get(API_URL, params=params)
  data = res.json()
  img_urls = data.get("images_results", [])

  next_url = str(data["serpapi_pagination"]["next"]) + f"&api_key={API_KEY}"
  res = requests.get(next_url)
  data = res.json()
  img_urls += data.get("images_results", [])

  for cnt,image in enumerate(img_urls):
    cnt_str = f"0000{cnt}"[-3:]
    try:
      img = image_from_url(image["original"]).convert("RGB")
    except:
      continue
    img.thumbnail((500, 500))
    img.save(f"dataset/train/{c_path}/{c_path}_{cnt_str}.jpg")
    sleep(0.2)

### Treinamento / Ajuste

In [ ]:
DATASET_PATH = "dataset"

In [ ]:
model = YOLO("yolo26m-cls.pt")

In [ ]:
model.train(data=DATASET_PATH, trainer=CustomizedTrainer, epochs=8, imgsz=128, batch=64, device=0)

In [ ]:
val_metrics = model.val(data=DATASET_PATH, validator=CustomizedValidator, imgsz=128, batch=64, device=0)
display(val_metrics.top1)

### Inferência

In [ ]:
img = PImage.open("./dataset/test/cebolinha/cebolinha_040.jpg").convert("RGB")

In [ ]:
res = model.predict(img, verbose=False)[0]
res.show()

In [ ]:
display(res.names[res.probs.top1])